In [ ]:
#!/usr/bin/env python3
"""
다나와 컴퓨터 주요부품 크롤러 v6 (이미지 포함, 전 카테고리)
- 가격비교 탭 전체 수집 (CPU 기준 약 533개)
- 이미지 URL 포함 (img.danuri.io, shrink=500:*)
- 이어하기 지원 (중단 후 셀 재실행 시 자동 재개)
- 출력: 원시데이터/{부품명}.csv
"""

import os, re, json, time, random, traceback
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

# ── CWD 고정 ──────────────────────────────────────────────
NOTEBOOK_DIR = r'C:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly'
os.chdir(NOTEBOOK_DIR)
print(f"[CWD] {os.getcwd()}")

# ══════════════════════════════════════════════════════════
#  카테고리 목록
# ══════════════════════════════════════════════════════════
CATEGORIES = [
    {"name": "쿨러_튜닝",   "cate": "11236855"},
]

# ══════════════════════════════════════════════════════════
#  설정
# ══════════════════════════════════════════════════════════
OUTPUT_DIR      = "원시데이터"
HEADLESS        = True
SAVE_EVERY      = 10
FRESH           = False           # True: 처음부터 / False: 이어하기
EXCLUDE_KEYS    = {"유통회사", "등록년월", "안전확인인증", "적합성평가인증"}
DELAY_LIST_PAGE = (2.5, 4.0)
DELAY_DETAIL    = (1.5, 3.0)
DELAY_CATEGORY  = (5.0, 9.0)

IMG_SKIP_PATTERNS = ("noimg", "noprice", "loading", "blank", "data:image")


# ══════════════════════════════════════════════════════════
#  유틸
# ══════════════════════════════════════════════════════════
def ts():        return datetime.now().strftime("%H:%M:%S")
def log(msg, indent=0): print(f"[{ts()}] {'  '*indent}{msg}", flush=True)
def safe_name(s): return re.sub(r'[\\/:*?"<>|]', "_", s)

def slp(lo_hi, label=""):
    t = random.uniform(*lo_hi)
    if label: log(f"⏳ {label} ({t:.1f}초)", 2)
    time.sleep(t)

def progress_line(done, total, width=32):
    if total == 0: return "완료(0개)/전체(0개)"
    filled = int(width * done / total)
    bar = "█" * filled + "░" * (width - filled)
    pct = 100 * done / total
    return f"[{bar}] {pct:5.1f}%  완료({done}개)/전체({total}개)"

def normalize_img_url(raw: str) -> str:
    if not raw:
        return ""
    if raw.startswith("//"):
        raw = "https:" + raw
    elif raw.startswith("/"):
        raw = "https://prod.danawa.com" + raw
    lower = raw.lower()
    if any(p in lower for p in IMG_SKIP_PATTERNS):
        return ""
    raw = re.sub(r"shrink=[^&]+", "shrink=500:*", raw)
    return raw

def extract_img(tag) -> str:
    if tag is None:
        return ""
    raw = (tag.get("data-original") or
           tag.get("data-src") or
           tag.get("src") or "")
    return normalize_img_url(raw)


# ══════════════════════════════════════════════════════════
#  체크포인트
# ══════════════════════════════════════════════════════════
def ckpt_path(name):
    return os.path.join(OUTPUT_DIR, f"{safe_name(name)}_ckpt.json")

def load_ckpt(name):
    p = ckpt_path(name)
    log(f"체크포인트: {os.path.abspath(p)}", 1)
    if os.path.exists(p):
        try:
            with open(p, encoding="utf-8") as f:
                d = json.load(f)
            done  = len(d.get("done_pcodes", []))
            total = len(d.get("items", []))
            log(f"이어하기 → 완료({done}개)/전체({total}개) "
                f"| 목록수집={'완료' if d.get('list_done') else '미완'}", 1)
            return d
        except Exception as e:
            log(f"체크포인트 읽기 실패({e}), 처음부터 시작", 1)
    else:
        log("체크포인트 없음 → 처음부터 시작", 1)
    return {"list_done": False, "items": [], "done_pcodes": [], "rows": [], "spec_keys": []}

def save_ckpt(name, d):
    p = ckpt_path(name); tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False)
    os.replace(tmp, p)

def del_ckpt(name):
    p = ckpt_path(name)
    if os.path.exists(p): os.remove(p)


# ══════════════════════════════════════════════════════════
#  드라이버
# ══════════════════════════════════════════════════════════
def init_driver(headless=True):
    opts = webdriver.ChromeOptions()
    if headless: opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--lang=ko-KR")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    svc    = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=svc, options=opts)
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"}
    )
    driver.set_page_load_timeout(30)
    return driver


# ══════════════════════════════════════════════════════════
#  STEP 1 — 목록 파싱 (제품명 / 가격 / 이미지 / pcode)
# ══════════════════════════════════════════════════════════
IMG_SELECTORS = [
    ".prod_img img",
    ".img_area img",
    ".thumb_image img",
    ".prod_main_info img",
    "a > img",
    "img[src*='danuri']",
    "img[src*='danawa']",
    "img",
]

def pick_product_img(item) -> str:
    for sel in IMG_SELECTORS:
        url = extract_img(item.select_one(sel))
        if url:
            return url
    return ""

def parse_list_page(soup):
    products, seen = [], set()
    for item in soup.select(".prod_item"):
        try:
            classes = " ".join(item.get("class") or [])
            if any(x in classes for x in ("product_ad", "ad_prod")):
                continue

            title_tag = item.select_one(".prod_name a")
            if not title_tag: continue
            title = title_tag.get_text(strip=True)
            if not title: continue

            pcode = None
            href  = title_tag.get("href", "")
            if "pcode=" in href:
                pcode = href.split("pcode=")[1].split("&")[0]
            if not pcode:
                m = re.search(r"productItem(\d+)", item.get("id", ""))
                if m: pcode = m.group(1)
            if not pcode or pcode in seen: continue

            price_tag = (item.select_one(".price_sect strong") or
                         item.select_one(".price_sect em") or
                         item.select_one(".low_price strong"))
            if not price_tag: continue
            price = re.sub(r"[^\d]", "", price_tag.get_text(strip=True))
            if not price: continue

            img_url = pick_product_img(item)

            seen.add(pcode)
            products.append({
                "제품명": title, "가격": price,
                "이미지": img_url, "pcode": pcode
            })
        except Exception:
            continue
    return products


# ══════════════════════════════════════════════════════════
#  페이지 이동
# ══════════════════════════════════════════════════════════
def current_list_state(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    codes = [item.get("id", "").replace("productItem", "")
             for item in soup.select("#productListArea .prod_item[id^='productItem']")]
    now = soup.select_one(".prod_num_nav .num.now_on, .number_wrap .num.now_on")
    now_page = now.get_text(strip=True) if now else ""
    return now_page, codes[:5]

def wait_page_changed(driver, before_codes, next_page, timeout=12):
    end = time.time() + timeout
    while time.time() < end:
        now_page, codes = current_list_state(driver)
        if codes and codes != before_codes and (not now_page or now_page == str(next_page)):
            return True
        time.sleep(0.3)
    return False

def click_next_page(driver, current_page):
    next_page = current_page + 1

    before_page, before_codes = current_list_state(driver)

    # 방법 1: 다나와의 실제 동적 페이지 함수 호출
    try:
        moved = driver.execute_script(
            "if (typeof movePage === 'function') { movePage(arguments[0]); return true; } return false;",
            next_page
        )
        if moved and wait_page_changed(driver, before_codes, next_page):
            return True
    except Exception:
        pass

    # 방법 2: movePage onclick
    try:
        xpath = f'//div[contains(@class,"prod_num_nav")]//a[contains(@onclick,"movePage({next_page})") or normalize-space(.)="{next_page}"]'
        btn   = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].click();", btn)
        if wait_page_changed(driver, before_codes, next_page):
            return True
    except NoSuchElementException:
        pass

    # 방법 3: 숫자 버튼
    try:
        for btn in driver.find_elements(
                By.CSS_SELECTOR, ".prod_num_nav a.num, .number_wrap a.num, div.pageing a, .paging_number a, .number a"):
            if btn.text.strip() == str(next_page):
                driver.execute_script("arguments[0].click();", btn)
                if wait_page_changed(driver, before_codes, next_page):
                    return True
    except Exception:
        pass

    # 방법 4: 다음 페이지 묶음 버튼(10, 20페이지 이후)
    try:
        nb = driver.find_element(
            By.XPATH,
            '//a[contains(@class,"btn_next")] | //a[contains(@class,"nav_next")] | //a[contains(@class,"edge_nav")] | //a[contains(@onclick,"nextPage")]'
        )
        driver.execute_script("arguments[0].click();", nb)
        time.sleep(1.0)
        for btn in driver.find_elements(
                By.CSS_SELECTOR, ".prod_num_nav a.num, .number_wrap a.num, div.pageing a, .paging_number a, .number a"):
            if btn.text.strip() == str(next_page):
                driver.execute_script("arguments[0].click();", btn)
                if wait_page_changed(driver, before_codes, next_page):
                    return True
        if wait_page_changed(driver, before_codes, next_page):
            return True
    except NoSuchElementException:
        pass
    except Exception:
        pass

    log("마지막 페이지 도달", 2)
    return False


def collect_all_products(driver, cate):
    url = f"https://prod.danawa.com/list/?cate={cate}&sort=12"
    log(f"목록 페이지 로드: {url}", 1)
    driver.get(url)
    time.sleep(4)

    all_products, seen_pcodes, page = [], set(), 1
    while True:
        log(f"페이지 {page} 파싱 중...", 2)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)

        soup          = BeautifulSoup(driver.page_source, "html.parser")
        page_products = parse_list_page(soup)
        new           = [p for p in page_products if p["pcode"] not in seen_pcodes]
        for p in new:
            seen_pcodes.add(p["pcode"])
            all_products.append(p)

        log(f"페이지 {page}: {len(page_products)}개 | "
            f"신규 {len(new)}개 | 누계 {len(all_products)}개", 2)

        if not new and page > 1:
            log("신규 제품 없음 → 목록 수집 완료", 2)
            break
        if not click_next_page(driver, page):
            break
        slp(DELAY_LIST_PAGE)
        page += 1

    return all_products


# ══════════════════════════════════════════════════════════
#  STEP 2 — 상세 스펙 + 이미지 최종 보완
# ══════════════════════════════════════════════════════════
DETAIL_IMG_SELECTORS = [
    ".prod_img .thumb_img img",
    ".main_prodimg img",
    "#detail_imageview img",
    ".prod_photo img",
    ".img_wrap img",
    "img[src*='danuri']",
    "img[src*='danawa']",
]

def get_detail_info(driver, pcode, fallback_img=""):
    url = f"https://prod.danawa.com/info/?pcode={pcode}"
    for attempt in range(1, 4):
        try:
            driver.get(url)
            time.sleep(3)
            soup  = BeautifulSoup(driver.page_source, "html.parser")
            specs = {}

            # 이미지 — 목록에서 못 가져온 경우만 보완
            img_url = fallback_img
            if not img_url:
                for sel in DETAIL_IMG_SELECTORS:
                    url_cand = extract_img(soup.select_one(sel))
                    if url_cand:
                        img_url = url_cand
                        break

            # 스펙 테이블
            for table in soup.select(".spec_tbl, .prod_spec"):
                for row in table.select("tr"):
                    ths, tds = row.select("th"), row.select("td")
                    if ths and not tds:
                        for th in ths:
                            k = th.get_text(strip=True)
                            if k and k not in specs: specs[k] = "정보없음"
                        continue
                    for th, td in zip(ths, tds):
                        k = th.get_text(strip=True)
                        for a in td.find_all("a"): a.replace_with(a.get_text())
                        v = re.sub(r"\s+", " ", td.get_text(" ", strip=True)).strip()
                        if k and v and k not in specs: specs[k] = v

            specs = {k: v for k, v in specs.items() if k not in EXCLUDE_KEYS}
            return specs, img_url

        except WebDriverException as e:
            log(f"상세 로드 실패 (시도 {attempt}/3, pcode={pcode}): {type(e).__name__}", 3)
            if attempt < 3: time.sleep(random.uniform(3, 6))

    return {}, fallback_img


# ══════════════════════════════════════════════════════════
#  CSV 저장
# ══════════════════════════════════════════════════════════
BASE_COLS = ["제품명", "가격", "이미지"]

def save_csv(path, rows, spec_keys):
    if not rows: return
    cols = BASE_COLS + [k for k in spec_keys if k not in BASE_COLS]
    df   = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns: df[c] = ""
    tmp = path + ".tmp"
    df[cols].to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, path)


# ══════════════════════════════════════════════════════════
#  실행
# ══════════════════════════════════════════════════════════
os.makedirs(OUTPUT_DIR, exist_ok=True)

print()
print("=" * 62)
print("  다나와 컴퓨터 주요부품 크롤러 (이미지 포함, 전 카테고리)")
print(f"  시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  카테고리: {len(CATEGORIES)}개")
print(f"  출력 위치: {os.path.abspath(OUTPUT_DIR)}")
print(f"  모드: {'처음부터 (FRESH=True)' if FRESH else '이어하기 (FRESH=False)'}")
print("=" * 62)

log("Chrome 드라이버 초기화 중...")
driver = init_driver(headless=HEADLESS)
log("드라이버 준비 완료.")

saved = []
try:
    for cat_idx, CATEGORY in enumerate(CATEGORIES, 1):
        name     = CATEGORY["name"]
        cate     = CATEGORY["cate"]
        csv_path = os.path.join(OUTPUT_DIR, f"{safe_name(name)}.csv")

        print()
        print("━" * 62)
        log(f"[{cat_idx}/{len(CATEGORIES)}] {name}  →  {csv_path}")
        print("━" * 62)

        ckpt = ({"list_done": False, "items": [], "done_pcodes": [], "rows": [], "spec_keys": []}
                if FRESH else load_ckpt(name))
        done_set  = set(ckpt["done_pcodes"])
        all_items = ckpt["items"]
        rows      = ckpt["rows"]
        spec_keys = ckpt["spec_keys"]

        try:
            # ── STEP 1: 목록 수집 ─────────────────────────────────
            if not ckpt["list_done"]:
                print()
                print("─" * 62)
                log("STEP 1 — 목록 페이지 전체 수집")
                all_items = collect_all_products(driver, cate)
                if not all_items:
                    log(f"❌ [{name}] 수집된 제품이 없습니다. 다음 카테고리로 이동.")
                    continue
                ckpt.update({"items": all_items, "list_done": True})
                save_ckpt(name, ckpt)
                log(f"✔ 목록 수집 완료: 총 {len(all_items)}개")
            else:
                log(f"STEP 1 — 목록 수집 이미 완료 ({len(all_items)}개)")

            # ── STEP 2: 상세 수집 ─────────────────────────────────
            total   = len(all_items)
            pending = [it for it in all_items if it["pcode"] not in done_set]
            done_n  = total - len(pending)

            print()
            print("─" * 62)
            log("STEP 2 — 상세 스펙 & 이미지 수집")
            log(f"완료({done_n}개)/전체({total}개)  |  남은 {len(pending)}개")
            print("─" * 62)
            print()

            for idx, item in enumerate(pending, 1):
                global_idx = done_n + idx
                bar        = progress_line(global_idx, total)
                short_name = item["제품명"][:28]

                print(f"\r  {bar}  {short_name:<28}", end="", flush=True)

                specs, img_url = get_detail_info(
                    driver, item["pcode"], fallback_img=item.get("이미지", ""))

                row = {"제품명": item["제품명"], "가격": item["가격"], "이미지": img_url}
                row.update(specs)
                rows.append(row)

                for k in specs:
                    if k not in spec_keys: spec_keys.append(k)
                done_set.add(item["pcode"])

                if idx % SAVE_EVERY == 0 or idx == len(pending):
                    print()
                    ckpt.update({
                        "done_pcodes": list(done_set),
                        "rows":        rows,
                        "spec_keys":   spec_keys,
                    })
                    save_ckpt(name, ckpt)
                    save_csv(csv_path, rows, spec_keys)
                    print(f"  ── 중간 저장 [{ts()}]  "
                          f"완료({global_idx}개)/전체({total}개)"
                          f"  →  {os.path.basename(csv_path)}")
                    print()

                if idx < len(pending):
                    slp(DELAY_DETAIL)

            # ── 카테고리 완료 ──────────────────────────────────────
            print()
            save_csv(csv_path, rows, spec_keys)
            del_ckpt(name)
            saved.append(csv_path)

            print()
            print("=" * 62)
            print(f"  ✔  [{name}] 크롤링 완료!")
            print(f"  완료({total}개)/전체({total}개)")
            print(f"  저장: {os.path.abspath(csv_path)}")
            print("=" * 62)

        except KeyboardInterrupt:
            raise
        except Exception as e:
            print()
            log(f"❌ [{name}] 오류: {e}", 1)
            traceback.print_exc()
            log("다음 카테고리로 계속...", 1)

        # 카테고리 간 대기 (마지막 카테고리 제외)
        if cat_idx < len(CATEGORIES):
            slp(DELAY_CATEGORY, "다음 카테고리 대기")

except KeyboardInterrupt:
    print()
    log("🛑 중단됨 — 체크포인트가 저장되어 있습니다.")
    log("FRESH = False 로 재실행하면 이어서 진행됩니다.")
finally:
    try: driver.quit()
    except Exception: pass
    log("드라이버 종료.")

print()
print("=" * 62)
print(f"  종료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
if saved:
    print("  저장된 파일:")
    for f in saved:
        print(f"    ✔ {f}")
print("=" * 62)


[CWD] c:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly

  다나와 컴퓨터 주요부품 크롤러 (이미지 포함, 전 카테고리)
  시작: 2026-06-01 19:46:32
  카테고리: 1개
  출력 위치: c:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly\원시데이터
  모드: 이어하기 (FRESH=False)
[19:46:32] Chrome 드라이버 초기화 중...
[19:46:34] 드라이버 준비 완료.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[19:46:34] [1/1] 쿨러_튜닝  →  원시데이터\쿨러_튜닝.csv
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[19:46:34]   체크포인트: c:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly\원시데이터\쿨러_튜닝_ckpt.json
[19:46:34]   체크포인트 없음 → 처음부터 시작

──────────────────────────────────────────────────────────────
[19:46:34] STEP 1 — 목록 페이지 전체 수집
[19:46:34]   목록 페이지 로드: https://prod.danawa.com/list/?cate=11236855&sort=12
[19:47:05]     페이지 1 파싱 중...
[19:47:08]     페이지 1: 34개 | 신규 34개 | 누계 34개
[19:47:09]     마지막 페이지 도달
[19:47:09] ✔ 목록 수집 완료: 총 34개

──────────────────────────────────────────────────────────────
[19:47:09] STEP 2 — 상세 스펙 & 이미지 수집
[19:47:09] 완료(0개)/전체(34개)  |  남은 34개
────────────

In [ ]:
#!/usr/bin/env python3
"""
다나와 CPU 크롤러 v2
- 가격비교 항목 전체 수집 (약 533개)
- 이미지 URL 포함 (img.danuri.io, shrink=500:*)
- 이어하기 지원 (중단 후 셀 재실행 시 자동 재개)
- 출력: 가공데이터/CPU.csv
"""

import os, re, json, time, random, traceback
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

# ── CWD 고정 ──────────────────────────────────────────────
NOTEBOOK_DIR = r'C:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly'
os.chdir(NOTEBOOK_DIR)
print(f"[CWD] {os.getcwd()}")

# ══════════════════════════════════════════════════════════
#  설정
# ══════════════════════════════════════════════════════════
OUTPUT_DIR      = "가공데이터"
HEADLESS        = True
SAVE_EVERY      = 10
EXCLUDE_KEYS    = {"유통회사", "등록년월", "안전확인인증", "적합성평가인증"}
DELAY_LIST_PAGE = (2.5, 4.0)
DELAY_DETAIL    = (1.5, 3.0)

CATEGORY = {"name": "CPU", "cate": "112747"}

# 이미지 플레이스홀더 키워드
IMG_SKIP_PATTERNS = ("noimg", "noprice", "loading", "blank", "data:image")


# ══════════════════════════════════════════════════════════
#  유틸
# ══════════════════════════════════════════════════════════
def ts():        return datetime.now().strftime("%H:%M:%S")
def log(msg, indent=0): print(f"[{ts()}] {'  '*indent}{msg}", flush=True)
def safe_name(s): return re.sub(r'[\\/:*?"<>|]', "_", s)

def slp(lo_hi, label=""):
    t = random.uniform(*lo_hi)
    if label: log(f"⏳ {label} ({t:.1f}초)", 2)
    time.sleep(t)

def progress_line(done, total, width=32):
    if total == 0: return "완료(0개)/전체(0개)"
    filled = int(width * done / total)
    bar = "█" * filled + "░" * (width - filled)
    pct = 100 * done / total
    return f"[{bar}] {pct:5.1f}%  완료({done}개)/전체({total}개)"

def normalize_img_url(raw: str) -> str:
    """
    - 상대경로 → 절대경로
    - shrink=130:130 등 썸네일 파라미터 → shrink=500:*  (고화질)
    - 플레이스홀더면 빈 문자열 반환
    """
    if not raw:
        return ""
    if raw.startswith("//"):
        raw = "https:" + raw
    elif raw.startswith("/"):
        raw = "https://prod.danawa.com" + raw
    lower = raw.lower()
    if any(p in lower for p in IMG_SKIP_PATTERNS):
        return ""
    # shrink 파라미터 고화질로 교체
    raw = re.sub(r"shrink=[^&]+", "shrink=500:*", raw)
    return raw

def extract_img(tag) -> str:
    """img 태그에서 URL 추출 — lazy-load 대응 (data-original 우선)"""
    if tag is None:
        return ""
    raw = (tag.get("data-original") or
           tag.get("data-src") or
           tag.get("src") or "")
    return normalize_img_url(raw)


# ══════════════════════════════════════════════════════════
#  체크포인트
# ══════════════════════════════════════════════════════════
def ckpt_path(name):
    return os.path.join(OUTPUT_DIR, f"{safe_name(name)}_ckpt.json")

def load_ckpt(name):
    p = ckpt_path(name)
    log(f"체크포인트: {os.path.abspath(p)}", 1)
    if os.path.exists(p):
        try:
            with open(p, encoding="utf-8") as f:
                d = json.load(f)
            done  = len(d.get("done_pcodes", []))
            total = len(d.get("items", []))
            log(f"이어하기 → 완료({done}개)/전체({total}개) "
                f"| 목록수집={'완료' if d.get('list_done') else '미완'}", 1)
            return d
        except Exception as e:
            log(f"체크포인트 읽기 실패({e}), 처음부터 시작", 1)
    else:
        log("체크포인트 없음 → 처음부터 시작", 1)
    return {"list_done": False, "items": [], "done_pcodes": [], "rows": [], "spec_keys": []}

def save_ckpt(name, d):
    p = ckpt_path(name); tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False)
    os.replace(tmp, p)

def del_ckpt(name):
    p = ckpt_path(name)
    if os.path.exists(p): os.remove(p)


# ══════════════════════════════════════════════════════════
#  드라이버
# ══════════════════════════════════════════════════════════
def init_driver(headless=True):
    opts = webdriver.ChromeOptions()
    if headless: opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--lang=ko-KR")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    svc    = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=svc, options=opts)
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"}
    )
    driver.set_page_load_timeout(30)
    return driver


# ══════════════════════════════════════════════════════════
#  STEP 1 — 목록 파싱 (제품명 / 가격 / 이미지 / pcode)
# ══════════════════════════════════════════════════════════
# 이미지 셀렉터 우선순위 (다나와 HTML 구조 변경 대응)
IMG_SELECTORS = [
    ".prod_img img",
    ".img_area img",
    ".thumb_image img",
    ".prod_main_info img",
    "a > img",
    "img[src*='danuri']",
    "img[src*='danawa']",
    "img",
]

def pick_product_img(item) -> str:
    """prod_item 안에서 실제 제품 이미지 URL 추출"""
    for sel in IMG_SELECTORS:
        url = extract_img(item.select_one(sel))
        if url:
            return url
    return ""

def parse_list_page(soup):
    products, seen = [], set()
    for item in soup.select(".prod_item"):
        try:
            classes = " ".join(item.get("class") or [])
            if any(x in classes for x in ("product_ad", "ad_prod")):
                continue

            title_tag = item.select_one(".prod_name a")
            if not title_tag: continue
            title = title_tag.get_text(strip=True)
            if not title: continue

            # pcode
            pcode = None
            href  = title_tag.get("href", "")
            if "pcode=" in href:
                pcode = href.split("pcode=")[1].split("&")[0]
            if not pcode:
                m = re.search(r"productItem(\d+)", item.get("id", ""))
                if m: pcode = m.group(1)
            if not pcode or pcode in seen: continue

            # 가격 (가격비교 탭만)
            price_tag = (item.select_one(".price_sect strong") or
                         item.select_one(".price_sect em") or
                         item.select_one(".low_price strong"))
            if not price_tag: continue
            price = re.sub(r"[^\d]", "", price_tag.get_text(strip=True))
            if not price: continue

            # 이미지
            img_url = pick_product_img(item)

            seen.add(pcode)
            products.append({
                "제품명": title, "가격": price,
                "이미지": img_url, "pcode": pcode
            })
        except Exception:
            continue
    return products


# ══════════════════════════════════════════════════════════
#  페이지 이동
# ══════════════════════════════════════════════════════════
def click_next_page(driver, current_page):
    next_page = current_page + 1

    # 방법 1: URL 직접 이동
    try:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        page_nums = [a.get_text(strip=True)
                     for a in soup.select("div.pageing a, .paging_number a, .number a")]
        if str(next_page) not in page_nums:
            log("마지막 페이지 도달", 2)
            return False
        cur = driver.current_url
        new_url = (re.sub(r"page=\d+", f"page={next_page}", cur)
                   if "page=" in cur
                   else cur + ("&" if "?" in cur else "?") + f"page={next_page}")
        driver.get(new_url)
        time.sleep(0.8)
        return True
    except Exception:
        pass

    # 방법 2: movePage onclick
    try:
        xpath = f'//a[@onclick="javascript:movePage({next_page}); return false;"]'
        btn   = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.8)
        return True
    except NoSuchElementException:
        pass

    # 방법 3: 숫자 버튼
    try:
        for btn in driver.find_elements(
                By.CSS_SELECTOR, "div.pageing a, .paging_number a, .number a"):
            if btn.text.strip() == str(next_page):
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8)
                return True
    except Exception:
        pass

    log("마지막 페이지 도달", 2)
    return False


def collect_all_products(driver, cate):
    url = f"https://prod.danawa.com/list/?cate={cate}&sort=12"
    log(f"목록 페이지 로드: {url}", 1)
    driver.get(url)
    time.sleep(4)

    all_products, seen_pcodes, page = [], set(), 1
    while True:
        log(f"페이지 {page} 파싱 중...", 2)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)

        soup          = BeautifulSoup(driver.page_source, "html.parser")
        page_products = parse_list_page(soup)
        new           = [p for p in page_products if p["pcode"] not in seen_pcodes]
        for p in new:
            seen_pcodes.add(p["pcode"])
            all_products.append(p)

        log(f"페이지 {page}: {len(page_products)}개 | "
            f"신규 {len(new)}개 | 누계 {len(all_products)}개", 2)

        if not new and page > 1:
            log("신규 제품 없음 → 목록 수집 완료", 2)
            break
        if not click_next_page(driver, page):
            break
        slp(DELAY_LIST_PAGE)
        page += 1

    return all_products


# ══════════════════════════════════════════════════════════
#  STEP 2 — 상세 스펙 + 이미지 최종 보완
# ══════════════════════════════════════════════════════════
DETAIL_IMG_SELECTORS = [
    ".prod_img .thumb_img img",
    ".main_prodimg img",
    "#detail_imageview img",
    ".prod_photo img",
    ".img_wrap img",
    "img[src*='danuri']",
    "img[src*='danawa']",
]

def get_detail_info(driver, pcode, fallback_img=""):
    url = f"https://prod.danawa.com/info/?pcode={pcode}"
    for attempt in range(1, 4):
        try:
            driver.get(url)
            time.sleep(3)
            soup  = BeautifulSoup(driver.page_source, "html.parser")
            specs = {}

            # 이미지 — 목록에서 못 가져온 경우만 보완
            img_url = fallback_img
            if not img_url:
                for sel in DETAIL_IMG_SELECTORS:
                    url_cand = extract_img(soup.select_one(sel))
                    if url_cand:
                        img_url = url_cand
                        break

            # 스펙 테이블
            for table in soup.select(".spec_tbl, .prod_spec"):
                for row in table.select("tr"):
                    ths, tds = row.select("th"), row.select("td")
                    if ths and not tds:
                        for th in ths:
                            k = th.get_text(strip=True)
                            if k and k not in specs: specs[k] = "정보없음"
                        continue
                    for th, td in zip(ths, tds):
                        k = th.get_text(strip=True)
                        for a in td.find_all("a"): a.replace_with(a.get_text())
                        v = re.sub(r"\s+", " ", td.get_text(" ", strip=True)).strip()
                        if k and v and k not in specs: specs[k] = v

            specs = {k: v for k, v in specs.items() if k not in EXCLUDE_KEYS}
            return specs, img_url

        except WebDriverException as e:
            log(f"상세 로드 실패 (시도 {attempt}/3, pcode={pcode}): {type(e).__name__}", 3)
            if attempt < 3: time.sleep(random.uniform(3, 6))

    return {}, fallback_img


# ══════════════════════════════════════════════════════════
#  CSV 저장
# ══════════════════════════════════════════════════════════
BASE_COLS = ["제품명", "가격", "이미지"]

def save_csv(path, rows, spec_keys):
    if not rows: return
    cols = BASE_COLS + [k for k in spec_keys if k not in BASE_COLS]
    df   = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns: df[c] = ""
    tmp = path + ".tmp"
    df[cols].to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, path)


# ══════════════════════════════════════════════════════════
#  실행
# ══════════════════════════════════════════════════════════
os.makedirs(OUTPUT_DIR, exist_ok=True)

name     = CATEGORY["name"]
cate     = CATEGORY["cate"]
csv_path = os.path.join(OUTPUT_DIR, f"{name}.csv")

print()
print("=" * 62)
print(f"  다나와 {name} 크롤러  (이미지 포함, 이어하기 지원)")
print(f"  시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  저장: {os.path.abspath(csv_path)}")
print("=" * 62)

ckpt      = load_ckpt(name)
done_set  = set(ckpt["done_pcodes"])
all_items = ckpt["items"]
rows      = ckpt["rows"]
spec_keys = ckpt["spec_keys"]

log("Chrome 드라이버 초기화 중...")
driver = init_driver(headless=HEADLESS)
log("드라이버 준비 완료.")

try:
    # ── STEP 1: 목록 수집 ─────────────────────────────────
    if not ckpt["list_done"]:
        print()
        print("─" * 62)
        log("STEP 1 — 목록 페이지 전체 수집")
        all_items = collect_all_products(driver, cate)
        if not all_items:
            log("❌ 수집된 제품이 없습니다.")
            raise SystemExit
        ckpt.update({"items": all_items, "list_done": True})
        save_ckpt(name, ckpt)
        log(f"✔ 목록 수집 완료: 총 {len(all_items)}개")
    else:
        log(f"STEP 1 — 목록 수집 이미 완료 ({len(all_items)}개)")

    # ── STEP 2: 상세 수집 ─────────────────────────────────
    total   = len(all_items)
    pending = [it for it in all_items if it["pcode"] not in done_set]
    done_n  = total - len(pending)

    print()
    print("─" * 62)
    log("STEP 2 — 상세 스펙 & 이미지 수집")
    log(f"완료({done_n}개)/전체({total}개)  |  남은 {len(pending)}개")
    print("─" * 62)
    print()

    for idx, item in enumerate(pending, 1):
        global_idx = done_n + idx
        bar        = progress_line(global_idx, total)
        short_name = item["제품명"][:28]

        print(f"\r  {bar}  {short_name:<28}", end="", flush=True)

        specs, img_url = get_detail_info(
            driver, item["pcode"], fallback_img=item.get("이미지", ""))

        row = {"제품명": item["제품명"], "가격": item["가격"], "이미지": img_url}
        row.update(specs)
        rows.append(row)

        for k in specs:
            if k not in spec_keys: spec_keys.append(k)
        done_set.add(item["pcode"])

        if idx % SAVE_EVERY == 0 or idx == len(pending):
            print()
            ckpt.update({
                "done_pcodes": list(done_set),
                "rows":        rows,
                "spec_keys":   spec_keys,
            })
            save_ckpt(name, ckpt)
            save_csv(csv_path, rows, spec_keys)
            print(f"  ── 중간 저장 [{ts()}]  "
                  f"완료({global_idx}개)/전체({total}개)"
                  f"  →  {os.path.basename(csv_path)}")
            print()

        if idx < len(pending):
            slp(DELAY_DETAIL)

    # ── 완료 ──────────────────────────────────────────────
    print()
    save_csv(csv_path, rows, spec_keys)
    del_ckpt(name)

    print()
    print("=" * 62)
    print(f"  ✔  [{name}] 크롤링 완료!")
    print(f"  완료({total}개)/전체({total}개)")
    print(f"  저장: {os.path.abspath(csv_path)}")
    print(f"  종료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 62)

except KeyboardInterrupt:
    print()
    log("🛑 중단됨 — 체크포인트가 저장되어 있습니다.")
    log("이 셀을 다시 실행하면 이어서 진행됩니다.")
except SystemExit:
    pass
except Exception as e:
    print()
    log(f"❌ 오류: {e}")
    traceback.print_exc()
finally:
    try: driver.quit()
    except Exception: pass
    log("드라이버 종료.")


In [3]:
import os, sys, re, json, time, random, traceback
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

# ── 핵심 수정: 절대 경로로 CWD 고정 ──────────────────────────
NOTEBOOK_DIR = r'C:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly'
os.chdir(NOTEBOOK_DIR)
print(f"[CWD] {os.getcwd()}")

# ── 설정 ──────────────────────────────────────────────────
OUTPUT_DIR  = "."
HEADLESS    = True
SAVE_EVERY  = 10
EXCLUDE_KEYS = {"유통회사", "등록년월", "안전확인인증", "적합성평가인증"}
DELAY_LIST_PAGE = (2.5, 4.0)
DELAY_DETAIL    = (1.5, 3.0)
DELAY_CATEGORY  = (5.0, 9.0)

# SSD 부터 시작 (CPU·메인보드·RAM·그래픽카드는 이미 완료)
CATEGORIES_REMAIN = [
    {"name": "쿨러_튜닝",   "cate": "11236855"},
]

# ── 유틸 ──────────────────────────────────────────────────
def ts(): return datetime.now().strftime("%H:%M:%S")
def log(msg, indent=0): print(f"[{ts()}] {'  '*indent}{msg}", flush=True)
def slp(lo_hi, label=""):
    t = random.uniform(*lo_hi)
    if label: log(f"⏳ {label} ({t:.1f}초)", 2)
    time.sleep(t)
def safe_name(s): return re.sub(r'[\\/:*?"<>|]', "_", s)
def pbar(cur, total, w=40):
    if total == 0: return "[----] 0/0"
    f = int(w * cur / total)
    return f"[{'█'*f}{'░'*(w-f)}] {100*cur/total:5.1f}%  {cur}/{total}"

# ── 체크포인트 ─────────────────────────────────────────────
def ckpt_path(name): return os.path.join(OUTPUT_DIR, f"danawa_{safe_name(name)}_ckpt.json")

def load_ckpt(name):
    p = ckpt_path(name)
    log(f"체크포인트 경로: {os.path.abspath(p)}", 1)
    if os.path.exists(p):
        try:
            with open(p, encoding="utf-8") as f:
                d = json.load(f)
            log(f"체크포인트 로드 성공 → list_done={d.get('list_done')} /"
                f"완료={len(d.get('done_pcodes',[]))}개 / "
                f"전체={len(d.get('items',[]))}개", 1)
            return d
        except Exception as e:
            log(f"체크포인트 읽기 실패 ({e}), 처음부터 시작", 1)
    else:
        log("체크포인트 없음 → 처음부터 시작", 1)
    return {"list_done": False, "items": [], "done_pcodes": [], "rows": [], "spec_keys": []}

def save_ckpt(name, d):
    p = ckpt_path(name); tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False)
    os.replace(tmp, p)

def del_ckpt(name):
    p = ckpt_path(name)
    if os.path.exists(p): os.remove(p)

# ── 드라이버 ──────────────────────────────────────────────
def init_driver(headless=True):
    opts = webdriver.ChromeOptions()
    if headless: opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--lang=ko-KR")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)" \
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.\
    0.0.0 Safari/537.36")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"]
)
    opts.add_experimental_option("useAutomationExtension", False)
    svc = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=svc, options=opts)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"})
    driver.set_page_load_timeout(30)
    return driver

# ── 목록 파싱 ──────────────────────────────────────────────
def parse_list_page(soup):
    products, seen = [], set()
    for item in soup.select(".prod_item"):
        try:
            classes = " ".join(item.get("class") or [])
            if any(x in classes for x in ("product_ad", "ad_prod")): continue
            title_tag = item.select_one(".prod_name a")
            if not title_tag: continue
            title = title_tag.get_text(strip=True)
            if not title: continue
            pcode = None
            href = title_tag.get("href", "")
            if "pcode=" in href:
                pcode = href.split("pcode=")[1].split("&")[0]
            if not pcode:
                li_id = item.get("id", "")
                m = re.search(r"productItem(\d+)", li_id)
                if m: pcode = m.group(1)
            if not pcode or pcode in seen: continue
            price_tag = (item.select_one(".price_sect strong") or
                         item.select_one(".price_sect em") or
                         item.select_one(".low_price strong"))
            if not price_tag: continue
            price = re.sub(r"[^\d]", "", price_tag.get_text(strip=True))
            if not price: continue
            seen.add(pcode)
            products.append({"제품명": title, "가격": price, "pcode": pcode})
        except Exception: continue
    return products

def click_next_page(driver, current_page):
    next_page = current_page + 1
    try:
        xpath = f'//a[@onclick="javascript:movePage({next_page}); returnfalse;"]'
        btn = driver.find_element(By.XPATH, xpath)
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.8); return True
    except NoSuchElementException: pass
    try:
        for btn in driver.find_elements(By.CSS_SELECTOR, "div.pageing a,.paging_number a"):
            if btn.text.strip() == str(next_page):
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8); return True
    except Exception: pass
    try:
        nb = driver.find_element(By.XPATH,
            '//a[@class="btn_num btn_next"] | //a[contains(@class,"nav_next")] | '
            '//a[contains(@onclick,"nextPage")]')
        driver.execute_script("arguments[0].click();", nb)
        time.sleep(1.0)
        for btn in driver.find_elements(By.CSS_SELECTOR, "div.pageing a,.paging_number a"):
            if btn.text.strip() == str(next_page):
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8); return True
    except NoSuchElementException:
        log("마지막 페이지 도달", 2); return False
    except Exception as e:
        log(f"페이지 이동 실패: {e}", 2); return False
    return False

def collect_all_products(driver, cate):
    url = f"https://prod.danawa.com/list/?cate={cate}&sort=12"
    log(f"목록 로드: {url}", 1)
    driver.get(url); time.sleep(4)
    all_products, seen_pcodes, page = [], set(), 1
    while True:
        log(f"페이지 {page} 파싱 중...", 1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        page_products = parse_list_page(soup)
        new = [p for p in page_products if p["pcode"] not in seen_pcodes]
        for p in new:
            seen_pcodes.add(p["pcode"]); all_products.append(p)
        log(f"  → 이번 페이지 {len(page_products)}개 | 신규 {len(new)}개| 누계 {len(all_products)}개", 1)
        if not new and page > 1:
            log("신규 제품 없음 → 수집 완료", 1); break
        if not click_next_page(driver, page): break
        slp(DELAY_LIST_PAGE); page += 1
    return all_products

# ── 상세 스펙 ──────────────────────────────────────────────
def get_detail_info(driver, pcode):
    url = f"https://prod.danawa.com/info/?pcode={pcode}"
    for attempt in range(1, 4):
        try:
            driver.get(url); time.sleep(3)
            soup = BeautifulSoup(driver.page_source, "html.parser")
            specs = {}
            for table in soup.select(".spec_tbl, .prod_spec"):
                for row in table.select("tr"):
                    ths, tds = row.select("th"), row.select("td")
                    if ths and not tds:
                        for th in ths:
                            k = th.get_text(strip=True)
                            if k and k not in specs: specs[k] = "정보없음"
                        continue
                    for th, td in zip(ths, tds):
                        k = th.get_text(strip=True)
                        for a in td.find_all("a"): a.replace_with(a.get_text())
                        v = re.sub(r"\s+", " ", td.get_text(" ", strip=True)).strip()
                        if k and v and k not in specs: specs[k] = v
            return {k: v for k, v in specs.items() if k not in EXCLUDE_KEYS}
        except WebDriverException as e:
            log(f"상세 로드 실패 (시도 {attempt}/3, pcode={pcode}): {e}",
 3)
            if attempt < 3: time.sleep(random.uniform(3, 6))
    return {}

# ── CSV 저장 ───────────────────────────────────────────────
BASE_COLS = ["카테고리", "제품명", "가격"]

def save_csv(path, rows, spec_keys):
    if not rows: return
    cols = BASE_COLS + [k for k in spec_keys if k not in BASE_COLS]
    df = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns: df[c] = ""
    tmp = path + ".tmp"
    df[cols].to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, path)

# ── 카테고리 크롤링 ────────────────────────────────────────
def crawl_category(driver, cat, fresh=False):
    name = cat["name"]; cate = cat["cate"]
    csv_path = os.path.join(OUTPUT_DIR, f"danawa_{safe_name(name)}.csv")
    print(f"\n{'━'*65}")
    log(f"[{name}] 시작")
    print(f"{'━'*65}")

    ckpt = ({"list_done": False, "items": [], "done_pcodes": [], "rows":
[], "spec_keys": []}
            if fresh else load_ckpt(name))

    done_set  = set(ckpt["done_pcodes"])
    all_items = ckpt["items"]
    rows      = ckpt["rows"]
    spec_keys = ckpt["spec_keys"]

    if not ckpt["list_done"]:
        log("STEP 1: 목록 전체 페이지 수집", 1)
        all_items = collect_all_products(driver, cate)
        if not all_items:
            log("수집된 제품이 없습니다. 건너뜁니다.", 1); return None
        ckpt["items"] = all_items; ckpt["list_done"] = True
        save_ckpt(name, ckpt)
        log(f"목록 수집 완료: 총 {len(all_items)}개", 1)
    else:
        log(f"STEP 1: 목록 수집 이미 완료 ({len(all_items)}개)", 1)

    total   = len(all_items)
    pending = [it for it in all_items if it["pcode"] not in done_set]
    done_n  = total - len(pending)

    log("STEP 2: 상세 정보 수집", 1)
    log(f"전체 {total}개 | 완료 {done_n}개 | 남은 {len(pending)}개", 1)

    for idx, item in enumerate(pending, 1):
        global_idx = done_n + idx
        print(f"\r  {pbar(global_idx, total)}  {item['제품명'][:35]:<35}", end="", flush=True)
        specs = get_detail_info(driver, item["pcode"])
        row = {"카테고리": name, "제품명": item["제품명"], "가격": item["가격"]}
        row.update(specs)
        rows.append(row)
        for k in specs:
            if k not in spec_keys: spec_keys.append(k)
        done_set.add(item["pcode"])
        if idx % SAVE_EVERY == 0 or idx == len(pending):
            print()
            ckpt["done_pcodes"] = list(done_set)
            ckpt["rows"]        = rows
            ckpt["spec_keys"]   = spec_keys
            save_ckpt(name, ckpt)
            save_csv(csv_path, rows, spec_keys)
            log(f"중간 저장 완료 ({global_idx}/{total}개) → {csv_path}",2)
        if idx < len(pending): slp(DELAY_DETAIL)

    print()
    save_csv(csv_path, rows, spec_keys)
    del_ckpt(name)
    log(f"✔ [{name}] 완료!  {total}개 → {csv_path}", 1)
    return csv_path

# ══════════════════════════════════════════════════════════
#  실행: SSD 체크포인트 확인 후 SSD → HDD → 케이스 → 파워 → 쿨러
# ══════════════════════════════════════════════════════════
ckpt_file = os.path.join(OUTPUT_DIR, "danawa_SSD_ckpt.json")
print(f"\n체크포인트 파일: {os.path.abspath(ckpt_file)}")
print(f"파일 존재 여부: {os.path.exists(ckpt_file)}")
if os.path.exists(ckpt_file):
    with open(ckpt_file, encoding="utf-8") as _f:
        _d = json.load(_f)
    _done = len(_d.get("done_pcodes", []))
    _total = len(_d.get("items", []))
    print(f"진행 상황: {_done}/{_total} 완료, {_total - _done}개 남음\n")

log("Chrome 드라이버 초기화 중...")
driver = init_driver(headless=HEADLESS)
log("드라이버 준비 완료.")

saved = []
try:
    for i, cat in enumerate(CATEGORIES_REMAIN, 1):
        log(f"\n[{i}/{len(CATEGORIES_REMAIN)}] 카테고리: {cat['name']}")
        try:
            out = crawl_category(driver, cat, fresh=False)
            if out: saved.append(out)
        except KeyboardInterrupt:
            print(); log("🛑 Ctrl+C — 체크포인트 저장 완료. 이 셀 재실행시 이어서 진행됩니다.")
            raise
        except Exception as e:
            log(f"⚠ [{cat['name']}] 오류: {e}", 1)
            traceback.print_exc()
            log("다음 카테고리로 계속...", 1)
        if i < len(CATEGORIES_REMAIN):
            slp(DELAY_CATEGORY, "다음 카테고리 대기")
except KeyboardInterrupt:
    pass
finally:
    try: driver.quit()
    except Exception: pass
    log("드라이버 종료.")

print("\n저장된 파일:")
for f in saved: print(f"  ✔ {f}")



[CWD] c:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly

체크포인트 파일: c:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly\danawa_SSD_ckpt.json
파일 존재 여부: False
[10:44:05] Chrome 드라이버 초기화 중...
[10:44:08] 드라이버 준비 완료.
[10:44:08] 
[1/1] 카테고리: 쿨러_튜닝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[10:44:08] [쿨러_튜닝] 시작
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[10:44:08]   체크포인트 경로: c:\Users\82107\Desktop\파이썬 프로젝트\pc_assembly\danawa_쿨러_튜닝_ckpt.json
[10:44:08]   체크포인트 없음 → 처음부터 시작
[10:44:08]   STEP 1: 목록 전체 페이지 수집
[10:44:08]   목록 로드: https://prod.danawa.com/list/?cate=11236855&sort=12
[10:44:40]   페이지 1 파싱 중...
[10:44:43]     → 이번 페이지 32개 | 신규 32개| 누계 32개
[10:44:45]   목록 수집 완료: 총 32개
[10:44:45]   STEP 2: 상세 정보 수집
[10:44:45]   전체 32개 | 완료 0개 | 남은 32개
  [████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  31.2%  10/32  Thermalright Peerless Assassin 120 
[10:46:13]     중간 저장 완료 (10/32개) → .\danawa_쿨러_튜닝.csv
  [█████████████████████████░░░░░░░░░░░░░░░]  62.5%  20/32  쿨러마스터 Master

In [1]:
import pandas as pd 
df = pd.read_csv('cpu.csv')

df

FileNotFoundError: [Errno 2] No such file or directory: 'cpu.csv'

In [12]:
import pandas as pd
import ast

df = pd.read_csv('sdd_data_all.csv', encoding='utf-8-sig')

def create_embedding_text(row):
    specs = ast.literal_eval(row['상세정보'])
    spec_text = " ".join([f"{k} {v}" for k, v in specs.items()])
    return f"{row['제품명']} {row['가격']} {row['제조사']} {spec_text}"

df['텍스트'] = df.apply(create_embedding_text, axis=1)
df.to_csv('sdd.csv', index=False, encoding='utf-8-sig')

In [11]:
import pandas as pd 

df = pd.read_csv(r'C:\Users\PCN\Desktop\pcn\가공데이터\ssd.csv')
df = df.drop('상세정보',axis=1)
df.to_csv(r'C:\Users\PCN\Desktop\pcn\가공데이터\ssd.csv')

In [19]:
df = df.drop('상세정보',axis=1)
df


,제품명,가격,제조사,텍스트
0,AMD 라이젠7-5세대 7800X3D (라파엘) (멀티팩 정품),"453,640",AMD(제조사 웹사이트 바로가기),"AMD 라이젠7-5세대 7800X3D (라파엘) (멀티팩 정품) 453,640 AM..."
1,인텔 코어 울트라5 시리즈2 245K (애로우레이크) (정품),"320,910",인텔(제조사 웹사이트 바로가기),"인텔 코어 울트라5 시리즈2 245K (애로우레이크) (정품) 320,910 인텔(..."
2,인텔 코어 울트라7 시리즈2 265K (애로우레이크) (정품),"449,530",인텔(제조사 웹사이트 바로가기),"인텔 코어 울트라7 시리즈2 265K (애로우레이크) (정품) 449,530 인텔(..."
3,AMD 라이젠5-6세대 9600 (그래니트 릿지) (멀티팩 정품),"275,930",AMD(제조사 웹사이트 바로가기),"AMD 라이젠5-6세대 9600 (그래니트 릿지) (멀티팩 정품) 275,930 A..."
4,AMD 라이젠5-5세대 7500F (라파엘),"185,470",AMD(제조사 웹사이트 바로가기),"AMD 라이젠5-5세대 7500F (라파엘) 185,470 AMD(제조사 웹사이트 ..."
...,...,...,...,...
547,인텔 펜티엄 E5300 (울프데일),940,인텔 펜티엄,인텔 펜티엄 E5300 (울프데일) 940 인텔 펜티엄 인텔 CPU종류 인텔(펜티엄...
548,인텔 코어2듀오 E8500 (울프데일),"4,700",인텔 코어2듀오,"인텔 코어2듀오 E8500 (울프데일) 4,700 인텔 코어2듀오 인텔 CPU종류 ..."
549,인텔 셀러론 G1630 (아이비브릿지),"2,200",인텔 셀러론,"인텔 셀러론 G1630 (아이비브릿지) 2,200 인텔 셀러론 인텔 CPU종류 인텔..."
550,인텔 펜티엄 E2200 (콘로),"5,400",인텔 펜티엄,"인텔 펜티엄 E2200 (콘로) 5,400 인텔 펜티엄 인텔 CPU종류 인텔(펜티엄..."
